## INSTALL REQUIREMENTS (Colab)

In [1]:
pip install timm torch torchvision scikit-learn matplotlib seaborn

## Import Libraries

In [2]:

import os
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, f1_score,
    precision_score, recall_score,
    mean_squared_error, cohen_kappa_score
)

# -------------------------
# CONFIGURATION
# -------------------------
TRAIN_DIR = "/content/drive/MyDrive/Knee_OADataset/224ClaheOAI4class/train"
TEST_DIR  = "/content/drive/MyDrive/Knee_OADataset/224ClaheOAI4class/test"

IMG_SIZE = 6082
BATCH_SIZE = 64
EPOCHS = 80
LR = 1e-4
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["Healthy", "Mild", "Moderate", "Severe"]

# -------------------------
# DATA AUGMENTATION
# -------------------------
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9,1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

## DATA LOADERS


In [4]:

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)


## EfficientNetV2B3 MODEL


In [5]:
import timm
import torch.nn as nn

class KneeOA_Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "tf_efficientnetv2_b3",
            pretrained=True,
            num_classes=0   # removes original classifier
        )

        self.classifier = nn.Sequential(
            nn.Linear(1536, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

model = KneeOA_Model().to(DEVICE)

# -------------------------
# LOSS & OPTIMIZER
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/57.9M [00:00<?, ?B/s]

## TRAINING LOOP


In [ ]:

train_acc, val_acc = [], []
train_loss, val_loss = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()

    train_loss.append(running_loss/len(train_loader))
    train_acc.append(correct/len(train_dataset))

    # ------------------
    model.eval()
    running_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()

    val_loss.append(running_loss/len(test_loader))
    val_acc.append(correct/len(test_dataset))

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Acc: {train_acc[-1]:.4f} | "
          f"Val Acc: {val_acc[-1]:.4f}")

## SAVE MODEL


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Knee_OADataset/224ClaheOAI4class/knee_oa_efficientnetv2b3.pth")

## PLOT ACCURACY & LOSS


In [ ]:



plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_acc,label="Train")
plt.plot(val_acc,label="Val")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_loss,label="Train")
plt.plot(val_loss,label="Val")
plt.title("Loss")
plt.legend()

plt.show()


## PREDICTIONS


In [ ]:

y_true, y_pred, y_prob = [], [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs,1)

        y_true.extend(labels.numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

## Performance Metrics and COnfusion Matrix


In [ ]:


cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")
mse       = mean_squared_error(y_true, y_pred)
kappa     = cohen_kappa_score(y_true, y_pred)

specificity = np.mean(
    np.diag(cm) / (np.sum(cm,axis=1) + 1e-6)
)

print("\nPrecision:",precision)
print("Recall:",recall)
print("F1-score:",f1)
print("Specificity:",specificity)
print("MSE:",mse)
print("Kappa:",kappa)

print("\nClassification Report\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## ROC-AUC CURVE

In [ ]:

from sklearn.preprocessing import label_binarize
y_bin = label_binarize(y_true, classes=[0,1,2,3])

plt.figure(figsize=(7,6))

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{CLASS_NAMES[i]} AUC={roc_auc:.3f}")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()